In [2]:
# !pip install mirdata
import mirdata

In [3]:
from typing import BinaryIO, Optional, TextIO, Tuple

from deprecated.sphinx import deprecated
import librosa
import numpy as np

from mirdata import download_utils, jams_utils, core, io, annotations

In [4]:
INDEXES = {
    "default": "1.0",
    "test": "sample",
    "1.0": core.Index(
        filename="gtzan_genre_index_1.0.json",
        url="https://zenodo.org/records/13993311/files/gtzan_genre_index_1.0.json?download=1",
        checksum="533ca050855f22acf2feb283d9957fe3",
        partial_download=["all", "tempo_beat_annotations"],
    ),
    "mini": core.Index(
        filename="gtzan_genre_1.0_mini_index.json",
        url="https://zenodo.org/records/14004436/files/gtzan_genre_1.0_mini_index.json?download=1",
        checksum="ac97f5a783d7843cf92ed8875d85af3d",
        partial_download=["mini", "tempo_beat_annotations"],
    ),
    "sample": core.Index(filename="gtzan_genre_index_1.0_sample.json"),
}

In [5]:
import mirdata.datasets
import mirdata.datasets.gtzan_genre

class Track(core.Track):
    """gtzan_genre Track class

    Args:
        track_id (str): track id of the track

    Attributes:
        audio_path (str): path to the audio file
        genre (str): annotated genre
        track_id (str): track id

    Cached Properties:
        beats (BeatData): human-labeled beat annotations
        tempo (float): global tempo annotations

    """

    def __init__(self, track_id, data_home, dataset_name, index, metadata):
        super().__init__(track_id, data_home, dataset_name, index, metadata)

        self.genre = track_id.split(".")[0]
        if self.genre == "hiphop":
            self.genre = "hip-hop"

        self.audio_path = self.get_path("audio")
        self.beats_path = self.get_path("beats")
        self.tempo_path = self.get_path("tempo")

    @property
    def audio(self) -> Optional[Tuple[np.ndarray, float]]:
        """The track's audio

        Returns:
            * np.ndarray - audio signal
            * float - sample rate

        """
        return load_audio(self.audio_path)
    
def load_audio(fhandle: BinaryIO) -> Tuple[np.ndarray, float]:
    """Load a GTZAN audio file.

    Args:
        fhandle (str or file-like): File-like object or path to audio file

    Returns:
        * np.ndarray - the mono audio signal
        * float - The sample rate of the audio file

    """
    audio, sr = librosa.load(fhandle, sr=22050, mono=True)
    return audio, sr

class Dataset(core.Dataset):
    """
    The gtzan_genre dataset
    """

    def __init__(self, data_home=None, version="default"):
        super().__init__(
            data_home,
            version,
            name="gtzan_genre",
            track_class=Track,
            # bibtex=BIBTEX,
            indexes=INDEXES,
        )

In [6]:
gtzan = Dataset(data_home="GTZAN")

In [7]:
gtzan.download(partial_download=['index'])

INFO: Downloading ['index']. Index is being stored in /Users/nicholasboyko/.pyenv/versions/3.9.20/envs/dl-hw2/lib/python3.9/site-packages/mirdata/datasets/indexes
INFO: [index] downloading gtzan_genre_index_1.0.json
INFO: /Users/nicholasboyko/.pyenv/versions/3.9.20/envs/dl-hw2/lib/python3.9/site-packages/mirdata/datasets/indexes/gtzan_genre_index_1.0.json already exists and will not be downloaded. Rerun with force_overwrite=True to delete this file and force the download.


In [8]:
gtzan.track('jazz.00054').audio
# librosa.feature.mfcc(y=y, sr=sr)

/var/folders/k5/qz9qh3jx46dd01ydlzds97900000gn/T/ipykernel_2354/3584938975.py:54: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(fhandle, sr=22050, mono=True)
/Users/nicholasboyko/.pyenv/versions/dl-hw2/lib/python3.9/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


NoBackendError: 

In [20]:
shapes = []
for key, value in gtzan.load_tracks().items():
    key
    try:
        shapes.append(value.audio[0].shape[0])
    except:
        print(f'error with track {key}')

/var/folders/k5/qz9qh3jx46dd01ydlzds97900000gn/T/ipykernel_2354/3584938975.py:54: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(fhandle, sr=22050, mono=True)
/Users/nicholasboyko/.pyenv/versions/dl-hw2/lib/python3.9/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


error with track jazz.00054


In [22]:
print(min(shapes))

660000


In [30]:
gtzan_mfccs = []
gtzan_keys = []
for key, value in gtzan.load_tracks().items():
    try:
        y, sr = value.audio
        mfccs = librosa.feature.mfcc(y=y[:min(shapes)], sr=sr, hop_length=256, n_mfcc=20)
        gtzan_mfccs.append(mfccs.T[1:])
        gtzan_keys.append(key.split('.')[0])
    except:
        print(f'track {key} not read')
    
gtzan_mfccs = np.array(gtzan_mfccs)
gtzan_keys = np.array(gtzan_keys)

/var/folders/k5/qz9qh3jx46dd01ydlzds97900000gn/T/ipykernel_2354/3584938975.py:54: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(fhandle, sr=22050, mono=True)
/Users/nicholasboyko/.pyenv/versions/dl-hw2/lib/python3.9/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


track jazz.00054 not read


In [31]:
gtzan_data = [gtzan_mfccs, gtzan_keys]

In [32]:
import pickle
with open('gtzan_mfccs.pkl', 'wb') as f: pickle.dump(gtzan_data, f)

[1, 2, 3]
